# 02 — Exchange Rate Forecasting

This notebook selects a SARIMA configuration and compares naive, SARIMA, and Holt forecasts for Mondays.

This is an educational experiment, not financial advice.

## Plan

1. Load historical rates through the project data module.
2. Tune SARIMA on an initial training period.
3. Backtest one-step forecasts for later Mondays.
4. Compare all models on the same observations.
5. Export predictions, metrics, and model parameters for analysis.

## Setup and imports

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [ ]:
project_root = Path.cwd()

if not (project_root / "currency_explorer").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / ".env", override=True)
api_key = os.getenv("API_KEY")

if not api_key:
    raise RuntimeError("API_KEY is not configured in .env")

print("Project root:", project_root)
print("API key loaded:", bool(api_key))

In [ ]:
from currency_explorer.data import (
    get_country_currency,
    get_historical_rates,
)
from currency_explorer.forecasting import (
    backtest_monday_forecasts,
    calculate_metrics,
    find_best_sarima,
)

## Historical data

In [ ]:
country = "Germany"
target_currency = "USD"

country_info = get_country_currency(country, api_key)
rates_df = get_historical_rates(
    base_currency=country_info["currency"],
    target_currency=target_currency,
    days=730,
)

print("Country:", country_info["country"])
print(
    "Currency pair:",
    f"{country_info['currency']}/{target_currency}",
)
print("Observations:", len(rates_df))
rates_df.head()

In [ ]:
rates_df.info()
rates_df.describe()

## SARIMA tuning

Parameters are selected only on the initial period. Later Mondays remain unseen during tuning.

In [ ]:
tuning_size = 252
tuning_history = rates_df["rate"].iloc[:tuning_size]

sarima_search_results = find_best_sarima(tuning_history)
sarima_search_results.head(10)

In [ ]:
best_order = sarima_search_results.loc[0, "order"]
best_seasonal_order = sarima_search_results.loc[
    0,
    "seasonal_order",
]

print("Best order:", best_order)
print("Best seasonal order:", best_seasonal_order)

## Monday backtest

Naive uses the previous observation, Holt uses 20 observations, and SARIMA uses up to 252 observations.

In [ ]:
model_results = backtest_monday_forecasts(
    rates_df=rates_df,
    order=best_order,
    seasonal_order=best_seasonal_order,
    tuning_size=tuning_size,
    holt_window=20,
    sarima_window=252,
    minimum_sarima_history=60,
)

model_results.head()

In [ ]:
evaluation_df = model_results.dropna(
    subset=[
        "naive_prediction",
        "sarima_prediction",
        "holt_prediction",
    ]
).copy()

print("Evaluated Mondays:", len(evaluation_df))

In [ ]:
metrics = pd.DataFrame({
    "Naive": calculate_metrics(
        evaluation_df["actual_rate"],
        evaluation_df["naive_prediction"],
    ),
    "SARIMA": calculate_metrics(
        evaluation_df["actual_rate"],
        evaluation_df["sarima_prediction"],
    ),
    "Holt": calculate_metrics(
        evaluation_df["actual_rate"],
        evaluation_df["holt_prediction"],
    ),
}).T

metrics

## Export results for the analysis notebook

In [ ]:
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

evaluation_df.to_csv(
    results_dir / "predictions.csv",
    index=False,
)
metrics.to_csv(
    results_dir / "metrics.csv",
    index=True,
)

sarima_config = {
    "order": list(best_order),
    "seasonal_order": list(best_seasonal_order),
    "tuning_size": tuning_size,
    "country": country_info["country"],
    "base_currency": country_info["currency"],
    "target_currency": target_currency,
}

with open(
    results_dir / "sarima_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(sarima_config, file, indent=4)

print("Saved:")
for result_path in sorted(results_dir.iterdir()):
    print("-", result_path.name)